In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

test = pd.read_csv("data/idsc_dataset.csv", delimiter=";")

In [10]:
test["eqq"] = np.where(test["aero_metar"] == test["destino"], 1, 0)

In [11]:
test[['aero_metar', 'destino', 'eqq']]

,aero_metar,destino,eqq
0,SBSP,SBSP,1
1,SBCT,SBCT,1
2,SBPA,SBPA,1
3,SBSV,SBSV,1
4,SBGR,SBGR,1
...,...,...,...
40034,SBRF,SBRF,1
40035,SBSP,SBSP,1
40036,SBCF,SBCF,1
40037,SBGR,SBGR,1


In [12]:
test["data"] = pd.to_datetime(test["dt_dep"])

/tmp/ipykernel_954390/180993028.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test["data"] = pd.to_datetime(test["dt_dep"])


In [13]:
flight_throught = pd.read_csv("data/test_flight_througt.csv")

In [14]:
test = test.merge(flight_throught, on="flightid", how="left")

In [15]:
def checkCols(df):
    checkArr = ["flightid","origem","destino","target","linha","diaSemana","hora","esperas","mean_x","sum_x","mean_y","sum_y","mean","sum","Temperaturamean_x","Temperaturamin_x","Temperaturamax_x","Temperaturastd_x","Ponto de Orvalhomean_x","Ponto de Orvalhomin_x","Ponto de Orvalhomax_x","Ponto de Orvalhostd_x","Velocidade do Ventomean_x","Velocidade do Ventomin_x","Velocidade do Ventomax_x","Velocidade do Ventostd_x","Direção do Ventomean_x","Direção do Ventomin_x","Direção do Ventomax_x","Direção do Ventostd_x","Visibilidademean_x","Visibilidademin_x","Visibilidademax_x","Visibilidadestd_x","Pressãomean_x","Pressãomin_x","Pressãomax_x","Pressãostd_x","Temperaturamean_y","Temperaturamin_y","Temperaturamax_y","Temperaturastd_y","Ponto de Orvalhomean_y","Ponto de Orvalhomin_y","Ponto de Orvalhomax_y","Ponto de Orvalhostd_y","Velocidade do Ventomean_y","Velocidade do Ventomin_y","Velocidade do Ventomax_y","Velocidade do Ventostd_y","Direção do Ventomean_y","Direção do Ventomin_y","Direção do Ventomax_y","Direção do Ventostd_y","Visibilidademean_y","Visibilidademin_y","Visibilidademax_y","Visibilidadestd_y","Pressãomean_y","Pressãomin_y","Pressãomax_y","Pressãostd_y","troca","trocareal","troca_prev_meanb2","troca_prev_sumb2","troca_prev_meantcprev_sem","troca_prev_sumtcprev_sem","troca_realb5","troca_realtcreal_sem","troca_real","dt_radar_lambda","dt_radar_lambda1","dt_radar_lambda2","dt_radar_lambda3","flightlevel_mean","flightlevel_max","distancia_","flightlevel_std","flightlevel_std1","speed_std","speed_std1","sum(flight_through_area)"]

    for col in checkArr:
        if col not in df.columns:
            print(col)

In [16]:
test["data"] = pd.to_datetime(test["path"].str.split("/").str[-1].str.split("_", expand=True)[1].str.split(".", expand=True)[0])

In [17]:
test["diaSemana"] = test["data"].dt.dayofweek

In [18]:
test["linha"] = test["origem"] + test["destino"]
test["hora"] = test["data"].dt.hour


In [19]:
esperas_hora = pd.read_csv("data/esperas_hora.csv")
esperas_semana = pd.read_csv("data/esperas_semana.csv")
esperas_aero  = pd.read_csv("data/esperas_aero.csv")

In [20]:
print(len(test))
test = test.merge(esperas_hora, right_on=["aero", "hora"], left_on=["origem", "hora"] , how="left")
test = test.merge(esperas_semana, right_on=["aero", "diaSemana"], left_on=["origem", "diaSemana"] , how="left")
test = test.merge(esperas_aero, right_on=["aero"], left_on=["origem"] , how="left")
print(len(test))

40039
40039


In [21]:
metar = pd.read_csv("data/test_metar.csv")

In [22]:
import pandas as pd
from metar.Metar import Metar

# Função para parsear METAR
def parse_metar(metar):
    metar = metar.replace('METAF', 'METAR') 
    try:
        metar_obj = Metar(metar)
        temperature = metar_obj.temp.value() if metar_obj.temp else None
        dewpoint = metar_obj.dewpt.value() if metar_obj.dewpt else None
        wind_speed = metar_obj.wind_speed.value() if metar_obj.wind_speed else None
        wind_direction = metar_obj.wind_dir.value() if metar_obj.wind_dir else None
        visibility = metar_obj.vis.value() if metar_obj.vis else None
        pressure = metar_obj.press.value() if metar_obj.press else None
        weather = ', '.join([str(c) for c in metar_obj.weather]) if metar_obj.weather else None
        sky = ', '.join([str(c) for c in metar_obj.sky]) if metar_obj.sky else None
        return pd.Series([temperature, dewpoint, wind_speed, wind_direction, visibility, pressure, weather, sky])
    except Exception as e:
        #print(str(e))
        return pd.Series([None, None, None, None, None, None, None, None])

# Aplicar parse sobre METAR para cada linha
metar[['Temperatura', 'Ponto de Orvalho', 'Velocidade do Vento', 'Direção do Vento', 'Visibilidade', 'Pressão', 'Tempo', 'Céu']] = metar['metar'].apply(parse_metar)


/tmp/ipykernel_954390/1735850452.py:23: FutureWarning: Returning a DataFrame from Series.apply when the supplied function returns a Series is deprecated and will be removed in a future version.
  metar[['Temperatura', 'Ponto de Orvalho', 'Velocidade do Vento', 'Direção do Vento', 'Visibilidade', 'Pressão', 'Tempo', 'Céu']] = metar['metar'].apply(parse_metar)


In [23]:
metar.fillna(-99, inplace=True)

In [24]:

metar['HoraData'] = pd.to_datetime(metar['data']).dt.strftime('%Y-%m-%d %H')
# Mesclar os dataframes usando a coluna "HoraData"

In [25]:
test['HoraData'] = pd.to_datetime(test['data']).dt.strftime('%Y-%m-%d %H')

In [26]:
# Agrupando por flight_id
df_merge = pd.merge(test, metar, left_on=['destino', 'HoraData'], right_on=['aero_metar', 'HoraData'], how='left')

grouped_df = df_merge.groupby('flightid')

# Calculando as estatísticas
stats_df = grouped_df.agg({
    'Temperatura': ['mean', 'min', 'max', 'std'],
    'Ponto de Orvalho': ['mean', 'min', 'max', 'std'],
    'Velocidade do Vento': ['mean', 'min', 'max', 'std'],
    'Direção do Vento': ['mean', 'min', 'max', 'std'],
    'Visibilidade': ['mean', 'min', 'max', 'std'],
    'Pressão': ['mean', 'min', 'max', 'std']
})
stats_df.reset_index(inplace=True)
stats_df.columns = [''.join(col) for col in stats_df.columns]
test = test.merge(stats_df, on="flightid", how="left")

In [27]:

metaf = pd.read_csv("data/test_metaf.csv")
metaf['metaf'] = metaf['metaf'].apply(lambda x: ' '.join(x.split()))

metaf[['Temperatura', 'Ponto de Orvalho', 'Velocidade do Vento', 'Direção do Vento', 'Visibilidade', 'Pressão', 'Tempo', 'Céu']] = metaf['metaf'].apply(parse_metar)

test

/tmp/ipykernel_954390/334265695.py:4: FutureWarning: Returning a DataFrame from Series.apply when the supplied function returns a Series is deprecated and will be removed in a future version.
  metaf[['Temperatura', 'Ponto de Orvalho', 'Velocidade do Vento', 'Direção do Vento', 'Visibilidade', 'Pressão', 'Tempo', 'Céu']] = metaf['metaf'].apply(parse_metar)


,flightid,origem,destino,hora_ref,dt_dep,snapshot_radar,path,hora_esperas,esperas,aero_esperas,...,Direção do Ventomax,Direção do Ventostd,Visibilidademean,Visibilidademin,Visibilidademax,Visibilidadestd,Pressãomean,Pressãomin,Pressãomax,Pressãostd
0,4f0356600f61e3fcbea8ed8a137a2423,SBBR,SBSP,11:00:00,11:53:30,MULTIPOINT ((-0.9862962481402141 -0.2329259019...,http://satelite.cptec.inpe.br/repositoriogoes/...,10:00:00,1,SBSP,...,340.0,NaN,10000.0,10000.0,10000.0,NaN,1016.0,1016.0,1016.0,NaN
1,865dbbbe74bebea18a71f24342516ff0,SBSP,SBCT,00:00:00,00:04:37,MULTIPOINT ((-0.8863534722346053 -0.5093356150...,http://satelite.cptec.inpe.br/repositoriogoes/...,23:00:00,0,SBCT,...,130.0,NaN,10000.0,10000.0,10000.0,NaN,1020.0,1020.0,1020.0,NaN
2,1803af9cfc6a2c74d188481e3ffd848e,SBGR,SBPA,01:00:00,01:33:15,MULTIPOINT ((-0.8966138193125871 -0.5115119601...,http://satelite.cptec.inpe.br/repositoriogoes/...,00:00:00,0,SBPA,...,70.0,NaN,9000.0,9000.0,9000.0,NaN,1009.0,1009.0,1009.0,NaN
3,f6af733a687f904183efd149ec713be5,SBCT,SBSV,03:00:00,03:54:14,MULTIPOINT ((-0.8586718860112791 -0.4418648049...,http://satelite.cptec.inpe.br/repositoriogoes/...,02:00:00,0,SBSV,...,60.0,NaN,10000.0,10000.0,10000.0,NaN,1016.0,1016.0,1016.0,NaN
4,6117f9ac60b7f66b740c9130be433313,SBRJ,SBGR,00:00:00,00:27:32,MULTIPOINT ((-0.8914635585346382 -0.5234904405...,http://satelite.cptec.inpe.br/repositoriogoes/...,23:00:00,0,SBGR,...,60.0,NaN,10000.0,10000.0,10000.0,NaN,1013.0,1013.0,1013.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40034,4e08052066073297c7a3c55798618324,SBSV,SBRF,20:00:00,20:25:00,MULTIPOINT ((-0.8955415064673201 -0.5234137952...,http://satelite.cptec.inpe.br/repositoriogoes/...,19:00:00,0,SBRF,...,190.0,NaN,10000.0,10000.0,10000.0,NaN,1013.0,1013.0,1013.0,NaN
40035,e22f120c65f5759aa8df576b59d71be3,SBCT,SBSP,08:00:00,08:50:22,MULTIPOINT ((-0.8903194203987348 -0.5144905777...,http://satelite.cptec.inpe.br/repositoriogoes/...,07:00:00,0,SBSP,...,-99.0,NaN,10000.0,10000.0,10000.0,NaN,1019.0,1019.0,1019.0,NaN
40036,ca9399162037201a4d0c49c832a32739,SBGR,SBCF,11:00:00,11:30:03,MULTIPOINT ((-0.8963144777278773 -0.5234059876...,http://satelite.cptec.inpe.br/repositoriogoes/...,10:00:00,0,SBCF,...,160.0,NaN,9000.0,9000.0,9000.0,NaN,1018.0,1018.0,1018.0,NaN
40037,560136088f5f1493dbad809a3554172c,SBBR,SBGR,12:00:00,12:25:21,NaN,http://satelite.cptec.inpe.br/repositoriogoes/...,11:00:00,0,SBGR,...,30.0,NaN,10000.0,10000.0,10000.0,NaN,1014.0,1014.0,1014.0,NaN


In [28]:

metaf['HoraData'] = pd.to_datetime(metaf['data']).dt.strftime('%Y-%m-%d %H')
# Mesclar os dataframes usando a coluna "HoraData"


In [29]:
metaf

,aero_metaf,data,metaf,Temperatura,Ponto de Orvalho,Velocidade do Vento,Direção do Vento,Visibilidade,Pressão,Tempo,Céu,HoraData
0,SBBR,2023-08-15 01:00:00,METAF SBBR 150100Z 12005KT CAVOK 21/07 Q1019=,21.0,7.0,5.0,120.0,10000.0,1019.0,NaN,NaN,2023-08-15 01
1,SBBR,2023-08-15 02:00:00,METAF SBBR 150200Z 11005KT CAVOK 20/07 Q1019=,20.0,7.0,5.0,110.0,10000.0,1019.0,NaN,NaN,2023-08-15 02
2,SBBR,2023-08-15 03:00:00,METAF SBBR 150300Z 11004KT CAVOK 19/07 Q1018=,19.0,7.0,4.0,110.0,10000.0,1018.0,NaN,NaN,2023-08-15 03
3,SBBR,2023-08-15 07:00:00,METAF SBBR 150700Z 08007KT CAVOK 17/07 Q1017=,17.0,7.0,7.0,80.0,10000.0,1017.0,NaN,NaN,2023-08-15 07
4,SBBR,2023-08-15 08:00:00,METAF SBBR 150800Z 08008KT CAVOK 16/08 Q1018=,16.0,8.0,8.0,80.0,10000.0,1018.0,NaN,NaN,2023-08-15 08
...,...,...,...,...,...,...,...,...,...,...,...,...
7584,SBSP,2023-09-24 19:00:00,METAF SBSP 241900Z 17007KT CAVOK 34/14 Q1008=,34.0,14.0,7.0,170.0,10000.0,1008.0,NaN,NaN,2023-09-24 19
7585,SBSP,2023-09-24 20:00:00,METAF SBSP 242000Z 20013KT CAVOK 32/15 Q1009=,32.0,15.0,13.0,200.0,10000.0,1009.0,NaN,NaN,2023-09-24 20
7586,SBSP,2023-09-24 21:00:00,METAF SBSP 242100Z 16007KT CAVOK 32/13 Q1009=,32.0,13.0,7.0,160.0,10000.0,1009.0,NaN,NaN,2023-09-24 21
7587,SBSP,2023-09-24 22:00:00,METAF SBSP 242200Z 15004KT CAVOK 32/13 Q1010=,32.0,13.0,4.0,150.0,10000.0,1010.0,NaN,NaN,2023-09-24 22


In [30]:

df_merge = pd.merge(test, metaf, left_on=['destino', 'HoraData'], right_on=['aero_metaf', 'HoraData'], how='left')
# Agrupando por flight_id
grouped_df = df_merge.groupby('flightid')

# Calculando as estatísticas
stats_df = grouped_df.agg({
    'Temperatura': ['mean', 'min', 'max', 'std'],
    'Ponto de Orvalho': ['mean', 'min', 'max', 'std'],
    'Velocidade do Vento': ['mean', 'min', 'max', 'std'],
    'Direção do Vento': ['mean', 'min', 'max', 'std'],
    'Visibilidade': ['mean', 'min', 'max', 'std'],
    'Pressão': ['mean', 'min', 'max', 'std']
})
stats_df.columns = [''.join(col) for col in stats_df.columns]
stats_df.reset_index(inplace=True)

test = test.merge(stats_df, on="flightid", how="left")
# Preencher NaN na coluna de espera com 0 (sem espera)
test

,flightid,origem,destino,hora_ref,dt_dep,snapshot_radar,path,hora_esperas,esperas,aero_esperas,...,Direção do Ventomax_y,Direção do Ventostd_y,Visibilidademean_y,Visibilidademin_y,Visibilidademax_y,Visibilidadestd_y,Pressãomean_y,Pressãomin_y,Pressãomax_y,Pressãostd_y
0,4f0356600f61e3fcbea8ed8a137a2423,SBBR,SBSP,11:00:00,11:53:30,MULTIPOINT ((-0.9862962481402141 -0.2329259019...,http://satelite.cptec.inpe.br/repositoriogoes/...,10:00:00,1,SBSP,...,320.0,NaN,10000.0,10000.0,10000.0,NaN,1014.0,1014.0,1014.0,NaN
1,865dbbbe74bebea18a71f24342516ff0,SBSP,SBCT,00:00:00,00:04:37,MULTIPOINT ((-0.8863534722346053 -0.5093356150...,http://satelite.cptec.inpe.br/repositoriogoes/...,23:00:00,0,SBCT,...,110.0,NaN,1000.0,1000.0,1000.0,NaN,1020.0,1020.0,1020.0,NaN
2,1803af9cfc6a2c74d188481e3ffd848e,SBGR,SBPA,01:00:00,01:33:15,MULTIPOINT ((-0.8966138193125871 -0.5115119601...,http://satelite.cptec.inpe.br/repositoriogoes/...,00:00:00,0,SBPA,...,80.0,NaN,10000.0,10000.0,10000.0,NaN,1010.0,1010.0,1010.0,NaN
3,f6af733a687f904183efd149ec713be5,SBCT,SBSV,03:00:00,03:54:14,MULTIPOINT ((-0.8586718860112791 -0.4418648049...,http://satelite.cptec.inpe.br/repositoriogoes/...,02:00:00,0,SBSV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6117f9ac60b7f66b740c9130be433313,SBRJ,SBGR,00:00:00,00:27:32,MULTIPOINT ((-0.8914635585346382 -0.5234904405...,http://satelite.cptec.inpe.br/repositoriogoes/...,23:00:00,0,SBGR,...,320.0,NaN,10000.0,10000.0,10000.0,NaN,1010.0,1010.0,1010.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40034,4e08052066073297c7a3c55798618324,SBSV,SBRF,20:00:00,20:25:00,MULTIPOINT ((-0.8955415064673201 -0.5234137952...,http://satelite.cptec.inpe.br/repositoriogoes/...,19:00:00,0,SBRF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40035,e22f120c65f5759aa8df576b59d71be3,SBCT,SBSP,08:00:00,08:50:22,MULTIPOINT ((-0.8903194203987348 -0.5144905777...,http://satelite.cptec.inpe.br/repositoriogoes/...,07:00:00,0,SBSP,...,360.0,NaN,10000.0,10000.0,10000.0,NaN,1017.0,1017.0,1017.0,NaN
40036,ca9399162037201a4d0c49c832a32739,SBGR,SBCF,11:00:00,11:30:03,MULTIPOINT ((-0.8963144777278773 -0.5234059876...,http://satelite.cptec.inpe.br/repositoriogoes/...,10:00:00,0,SBCF,...,130.0,NaN,10000.0,10000.0,10000.0,NaN,1016.0,1016.0,1016.0,NaN
40037,560136088f5f1493dbad809a3554172c,SBBR,SBGR,12:00:00,12:25:21,NaN,http://satelite.cptec.inpe.br/repositoriogoes/...,11:00:00,0,SBGR,...,340.0,NaN,10000.0,10000.0,10000.0,NaN,1012.0,1012.0,1012.0,NaN


In [31]:
for col in test.columns:
    print(col)

flightid
origem
destino
hora_ref
dt_dep
snapshot_radar
path
hora_esperas
esperas
aero_esperas
hora_metaf
metaf
aero_metaf
hora_metar
metar
aero_metar
hora_tcp
troca
aero_tcp
hora_tcr
aero_tcr
eqq
data
first(substring(CAST(polygon AS STRING), 3, 13))
flight_through_area
diaSemana
linha
hora
aero_x
mean_x
sum_x
aero_y
mean_y
sum_y
aero
mean
sum
HoraData
Temperaturamean_x
Temperaturamin_x
Temperaturamax_x
Temperaturastd_x
Ponto de Orvalhomean_x
Ponto de Orvalhomin_x
Ponto de Orvalhomax_x
Ponto de Orvalhostd_x
Velocidade do Ventomean_x
Velocidade do Ventomin_x
Velocidade do Ventomax_x
Velocidade do Ventostd_x
Direção do Ventomean_x
Direção do Ventomin_x
Direção do Ventomax_x
Direção do Ventostd_x
Visibilidademean_x
Visibilidademin_x
Visibilidademax_x
Visibilidadestd_x
Pressãomean_x
Pressãomin_x
Pressãomax_x
Pressãostd_x
Temperaturamean_y
Temperaturamin_y
Temperaturamax_y
Temperaturastd_y
Ponto de Orvalhomean_y
Ponto de Orvalhomin_y
Ponto de Orvalhomax_y
Ponto de Orvalhostd_y
Velocidade do 

In [32]:
test[["hora_tcp" ,"troca" ,"aero_tcp" ,"hora_tcr" ,"aero_tcr"]]

,hora_tcp,troca,aero_tcp,hora_tcr,aero_tcr
0,11:00:00,0.0,SBSP,NaN,NaN
1,00:00:00,0.0,SBCT,23:00:00,SBCT
2,01:00:00,0.0,SBPA,NaN,NaN
3,03:00:00,0.0,SBSV,02:00:00,SBSV
4,00:00:00,0.0,SBGR,23:00:00,SBGR
...,...,...,...,...,...
40034,20:00:00,0.0,SBRF,NaN,NaN
40035,08:00:00,0.0,SBSP,NaN,NaN
40036,11:00:00,0.0,SBCF,NaN,NaN
40037,12:00:00,0.0,SBGR,NaN,NaN


In [33]:
resumo = pd.read_csv("data/resumo_voo.csv")

In [34]:
resumo

,linha_first,flightid_,dt_radar_<lambda>,dt_radar_<lambda>.1,dt_radar_<lambda>.2,dt_radar_<lambda>.3,flightlevel_mean,flightlevel_max,distancia_,flightlevel_std,flightlevel_std.1,speed_std,speed_std.1
0,NaN,count,mean,min,max,std,mean,mean,mean,std,mean,mean,std
1,SBBRSBCF,2644,3210.6637670196674,0.0,8277.0,506.9815754584058,223.23009635516672,352.7874432677761,575.6958980433018,15.414484379289922,113.11099040127081,96.26012004571069,16.617049986812884
2,SBBRSBCT,959,5524.459854014599,4.0,8166.0,602.8147149739482,281.071385902067,367.9061522419187,1061.908684010797,13.184672048521254,113.58039360634044,84.08358708763086,13.509180931157234
3,SBBRSBFL,315,6614.263492063492,4.0,14041.0,824.4473638920381,291.94466776671055,371.6634920634921,1282.3324062624968,15.232888859932347,115.89589711595092,79.03449466219763,14.21473759918302
4,SBBRSBGL,639,4499.599374021909,362.0,7138.0,539.795847627226,259.66979013256446,367.9154929577465,885.994544233408,13.252080132619195,122.03779859280792,92.35050625485451,15.93562582501301
...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,SBSVSBKP,1379,7368.569978245105,121.0,11225.0,761.98530512265,305.3446206649822,369.22770123277735,1426.4684886193365,17.341259072228155,102.26689170812047,68.5258508294501,16.681647582814303
134,SBSVSBPA,79,11771.683544303798,1683.0,13623.0,1455.2232848701617,324.43509942690906,375.1898734177215,2277.3825023041554,8.22824879734405,99.87921093160097,62.36285594137942,8.362795726130804
135,SBSVSBRF,1491,3784.3105298457413,0.0,9182.0,526.7623314570509,226.64509710011185,362.5083836351442,643.117568170169,15.707041969914366,127.34337283196844,97.04344098509462,14.56471447373063
136,SBSVSBRJ,1630,6706.908588957055,843.0,10924.0,630.3015703620225,291.5118729723326,378.08895705521473,1225.918106541467,12.702016763699218,123.87140594500069,82.84562463100339,13.215119734561947


In [35]:

test = test.merge(resumo, right_on="linha_first", left_on="linha", how="left")

In [36]:
test

,flightid,origem,destino,hora_ref,dt_dep,snapshot_radar,path,hora_esperas,esperas,aero_esperas,...,dt_radar_<lambda>.1,dt_radar_<lambda>.2,dt_radar_<lambda>.3,flightlevel_mean,flightlevel_max,distancia_,flightlevel_std,flightlevel_std.1,speed_std,speed_std.1
0,4f0356600f61e3fcbea8ed8a137a2423,SBBR,SBSP,11:00:00,11:53:30,MULTIPOINT ((-0.9862962481402141 -0.2329259019...,http://satelite.cptec.inpe.br/repositoriogoes/...,10:00:00,1,SBSP,...,4.0,10564.0,844.206687068207,269.5108682121877,372.4241692964544,834.6996540544909,19.009494181349424,114.8176237791538,87.14296232331128,20.13567366935704
1,865dbbbe74bebea18a71f24342516ff0,SBSP,SBCT,00:00:00,00:04:37,MULTIPOINT ((-0.8863534722346053 -0.5093356150...,http://satelite.cptec.inpe.br/repositoriogoes/...,23:00:00,0,SBCT,...,0.0,7381.0,334.8714333110137,164.23913260243307,276.8212482790271,334.25452103143675,9.689498360526402,87.12070781077232,81.32576898678269,14.762519911792358
2,1803af9cfc6a2c74d188481e3ffd848e,SBGR,SBPA,01:00:00,01:33:15,MULTIPOINT ((-0.8966138193125871 -0.5115119601...,http://satelite.cptec.inpe.br/repositoriogoes/...,00:00:00,0,SBPA,...,4.0,9122.0,722.3015969519972,254.84483420697651,351.0929075072584,865.2986184462269,17.465242901597914,116.24704154093632,76.29612120984584,15.92323623169146
3,f6af733a687f904183efd149ec713be5,SBCT,SBSV,03:00:00,03:54:14,MULTIPOINT ((-0.8586718860112791 -0.4418648049...,http://satelite.cptec.inpe.br/repositoriogoes/...,02:00:00,0,SBSV,...,7084.0,8944.0,552.3702919183285,328.9996589382752,383.3333333333333,1752.4578951347282,23.45196584695689,93.77715563512778,67.78345578170041,21.14963214107247
4,6117f9ac60b7f66b740c9130be433313,SBRJ,SBGR,00:00:00,00:27:32,MULTIPOINT ((-0.8914635585346382 -0.5234904405...,http://satelite.cptec.inpe.br/repositoriogoes/...,23:00:00,0,SBGR,...,0.0,13445.0,423.7949712878491,170.3935622592871,291.02193932827737,344.671384855944,9.565268551404948,93.36308388194044,76.7597576353726,12.79980170350178
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40034,4e08052066073297c7a3c55798618324,SBSV,SBRF,20:00:00,20:25:00,MULTIPOINT ((-0.8955415064673201 -0.5234137952...,http://satelite.cptec.inpe.br/repositoriogoes/...,19:00:00,0,SBRF,...,0.0,9182.0,526.7623314570509,226.64509710011185,362.5083836351442,643.117568170169,15.707041969914366,127.34337283196844,97.04344098509462,14.56471447373063
40035,e22f120c65f5759aa8df576b59d71be3,SBCT,SBSP,08:00:00,08:50:22,MULTIPOINT ((-0.8903194203987348 -0.5144905777...,http://satelite.cptec.inpe.br/repositoriogoes/...,07:00:00,0,SBSP,...,1.0,6902.0,490.2973280785355,165.95386924395322,276.2230528092597,335.4940716116764,10.369935771241394,86.37228534631143,96.1239264923841,16.4390436639803
40036,ca9399162037201a4d0c49c832a32739,SBGR,SBCF,11:00:00,11:30:03,MULTIPOINT ((-0.8963144777278773 -0.5234059876...,http://satelite.cptec.inpe.br/repositoriogoes/...,10:00:00,0,SBCF,...,0.0,7620.0,452.97491776260836,201.90803865094242,332.68273381294966,483.49518722815856,12.614499304145626,103.60776550195054,95.56959139292832,15.688727099511283
40037,560136088f5f1493dbad809a3554172c,SBBR,SBGR,12:00:00,12:25:21,NaN,http://satelite.cptec.inpe.br/repositoriogoes/...,11:00:00,0,SBGR,...,238.0,15064.0,636.8736573971083,263.464528117792,366.3066581306018,824.5630680169894,16.204102058420567,112.59666838840823,84.8848882045857,17.95215031368548


In [37]:
test_flight_through = pd.read_csv("data/test_flight_througt.csv")

In [38]:
test_flight_through["sum(flight_through_area)"] = test_flight_through["flight_through_area"]

In [39]:
test = test.merge(test_flight_through[["flightid", "sum(flight_through_area)"]], on="flightid", how="left")

In [40]:
#test.fillna(-99, inplace=True)

In [41]:
test["hora"] = test["hora"].astype(int)
test["diaSemana"] = test["diaSemana"].astype(int)

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [ ]:
for col in resumo.columns:
    print(col)  

linha_first
flightid_
diff_arr
diff_arr.1
diff_arr.2
diff_arr.3
diff_dep
diff_dep.1
diff_dep.2
diff_dep.3
dt_radar_<lambda_0>
dt_radar_<lambda_0>.1
dt_radar_<lambda_0>.2
dt_radar_<lambda_0>.3
flightlevel_mean
flightlevel_max
distancia_
flightlevel_std
flightlevel_std.1
speed_std
speed_std.1


In [ ]:
checkCols(test)

target
troca_prev_meanb2
troca_prev_sumb2
troca_prev_meantcprev_sem
troca_prev_sumtcprev_sem
troca_realb5
troca_realtcreal_sem
troca_real
dt_radar_lambda
dt_radar_lambda1
dt_radar_lambda2
dt_radar_lambda3


In [ ]:
import re
test.columns = [re.sub('[\[\]<>\'"\'.,]', '', col) for col in test.columns]

In [226]:
test.to_csv("data/test_with_flight_v2_tcfixed.csv", index=False)

In [87]:
def invertedCheckCols(df):
    checkArr = ["flightid","origem","destino","target","linha","diaSemana","hora","esperas","mean_x","sum_x","mean_y","sum_y","mean","sum","Temperaturamean_x","Temperaturamin_x","Temperaturamax_x","Temperaturastd_x","Ponto de Orvalhomean_x","Ponto de Orvalhomin_x","Ponto de Orvalhomax_x","Ponto de Orvalhostd_x","Velocidade do Ventomean_x","Velocidade do Ventomin_x","Velocidade do Ventomax_x","Velocidade do Ventostd_x","Direção do Ventomean_x","Direção do Ventomin_x","Direção do Ventomax_x","Direção do Ventostd_x","Visibilidademean_x","Visibilidademin_x","Visibilidademax_x","Visibilidadestd_x","Pressãomean_x","Pressãomin_x","Pressãomax_x","Pressãostd_x","Temperaturamean_y","Temperaturamin_y","Temperaturamax_y","Temperaturastd_y","Ponto de Orvalhomean_y","Ponto de Orvalhomin_y","Ponto de Orvalhomax_y","Ponto de Orvalhostd_y","Velocidade do Ventomean_y","Velocidade do Ventomin_y","Velocidade do Ventomax_y","Velocidade do Ventostd_y","Direção do Ventomean_y","Direção do Ventomin_y","Direção do Ventomax_y","Direção do Ventostd_y","Visibilidademean_y","Visibilidademin_y","Visibilidademax_y","Visibilidadestd_y","Pressãomean_y","Pressãomin_y","Pressãomax_y","Pressãostd_y","troca","dt_radar_lambda","dt_radar_lambda1","dt_radar_lambda2","dt_radar_lambda3","flightlevel_mean","flightlevel_max","distancia_","flightlevel_std","flightlevel_std1","speed_std","speed_std1","sum(flight_through_area)"]


    for col in df.columns:
        if col not in checkArr:
            print(col)

In [5]:
import pandas as pd
df2 = pd.read_csv("data/train_with_flight_v2.csv")

In [8]:
df2[df2["target"] < 1500]

,flightid,origem,destino,target,linha,diaSemana,hora,HoraDataDest,HoraData,esperas,...,dt_radar_lambda2,dt_radar_lambda3,flightlevel_mean,flightlevel_max,distancia_,flightlevel_std,flightlevel_std1,speed_std,speed_std1,sum(flight_through_area)
28,9e2aee75f2ad8a1f4f070861808eaea9,4,9,1443,22,2,10,2022-06-01 11,2022-06-01 10,3,...,1924.0,343.297314,72.378660,110.944444,80.379864,11.057367,29.635746,54.853758,13.772244,40.0
954,8b3f12d8ed9e058f3376775871eedb0b,8,11,-1364,70,3,7,2022-06-02 07,2022-06-02 07,0,...,20945.0,947.828544,309.615721,359.204056,2058.352275,14.780032,89.953260,62.034681,14.023980,45.0
1021,845430b3c2d634e4f28a9f21162368cc,11,0,1179,106,3,14,2022-06-02 14,2022-06-02 14,2,...,4864.0,716.457824,48.878891,77.575342,47.270524,12.550159,23.606039,50.220655,13.461787,30.0
1179,1ab0e0a1942895a7d531e0bfe66ac5b7,1,7,1076,109,3,13,2022-06-02 14,2022-06-02 13,5,...,4142.0,553.891634,71.093247,97.695431,45.222078,6.630996,22.678295,55.160932,12.330140,28.0
1288,aef3f7cf8b6978a497133ea80e0c86fc,4,2,-2585,41,3,17,2022-06-02 16,2022-06-02 17,0,...,8404.0,439.030271,186.246424,314.743395,482.810939,12.455830,104.402439,81.662949,14.570804,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256124,cdd4c1c5c94e36d0d628f01cbf881443,11,0,1106,106,5,16,2023-04-08 17,2023-04-08 16,1,...,4864.0,716.457824,48.878891,77.575342,47.270524,12.550159,23.606039,50.220655,13.461787,0.0
256443,bf0e736c2f902ee013835636ff036519,1,7,1191,109,6,19,2023-04-09 20,2023-04-09 19,0,...,4142.0,553.891634,71.093247,97.695431,45.222078,6.630996,22.678295,55.160932,12.330140,0.0
257830,e6402b4cc0f3640603f8ba51bc234ac1,11,0,833,106,0,8,2023-04-10 09,2023-04-10 08,0,...,4864.0,716.457824,48.878891,77.575342,47.270524,12.550159,23.606039,50.220655,13.461787,0.0
259964,e23be7130b140e2beb06866de0e60ffb,4,7,1072,129,3,19,2023-04-13 19,2023-04-13 19,5,...,5284.0,1605.835669,164.468150,244.111111,4.825028,48.946314,78.874241,82.498300,19.775066,0.0


In [45]:
df2.describe()

,origem,destino,target,linha,diaSemana,hora,esperas,mean_x,sum_x,mean_y,...,dt_radar_lambda2,dt_radar_lambda3,flightlevel_mean,flightlevel_max,distancia_,flightlevel_std,flightlevel_std1,speed_std,speed_std1,sum(flight_through_area)
count,262416.000000,262416.000000,262416.000000,262416.000000,262416.000000,262416.000000,262416.000000,262416.000000,262416.000000,262416.000000,...,262416.000000,262415.000000,262416.000000,262416.000000,262416.000000,262415.000000,262416.000000,262416.000000,262415.000000,253763.000000
mean,4.533241,5.403093,4630.257214,42.155486,2.862634,13.092933,1.764774,0.220402,71.410150,0.180670,...,10390.404682,583.240841,232.026657,336.125329,803.687152,13.435063,104.869539,83.837253,15.438774,33.529939
std,3.006866,3.675472,2532.894037,28.066757,1.950839,6.193750,4.163974,0.277428,89.886648,0.162126,...,3002.828131,163.504237,57.530248,42.873388,502.175201,3.176270,13.519327,11.132089,2.208574,66.125738
min,0.000000,0.000000,-84219.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003623,...,1923.000000,154.100351,48.878891,77.575342,4.825028,3.776107,20.968162,47.460946,5.375350,0.000000
25%,2.000000,2.000000,2962.000000,21.000000,1.000000,9.000000,0.000000,0.018519,6.000000,0.022163,...,8166.000000,478.524801,173.714559,293.436835,373.343995,10.830585,94.688262,76.759758,14.023980,0.000000
50%,4.000000,5.000000,4073.000000,34.000000,3.000000,13.000000,0.000000,0.077160,25.000000,0.167572,...,10022.000000,531.192062,240.871505,352.787443,643.117568,12.957144,104.402439,82.845625,15.365389,0.000000
75%,7.000000,8.000000,5382.000000,68.000000,4.000000,18.000000,2.000000,0.314815,102.000000,0.287234,...,12243.000000,695.101408,282.503820,370.824111,920.254067,15.414484,114.029671,93.166449,16.674661,57.000000
max,11.000000,11.000000,23470.000000,136.000000,6.000000,23.000000,82.000000,1.330247,431.000000,0.504529,...,20945.000000,2912.805349,339.095154,389.666667,2940.945547,70.885271,133.070262,113.037882,45.850905,446.000000


In [43]:
for col in df2.columns:
    print(col)

flightid
origem
destino
target
linha
diaSemana
hora
HoraDataDest
HoraData
esperas
mean_x
sum_x
mean_y
sum_y
mean
sum
Temperaturamean_x
Temperaturamin_x
Temperaturamax_x
Temperaturastd_x
Ponto de Orvalhomean_x
Ponto de Orvalhomin_x
Ponto de Orvalhomax_x
Ponto de Orvalhostd_x
Velocidade do Ventomean_x
Velocidade do Ventomin_x
Velocidade do Ventomax_x
Velocidade do Ventostd_x
Direção do Ventomean_x
Direção do Ventomin_x
Direção do Ventomax_x
Direção do Ventostd_x
Visibilidademean_x
Visibilidademin_x
Visibilidademax_x
Visibilidadestd_x
Pressãomean_x
Pressãomin_x
Pressãomax_x
Pressãostd_x
Temperaturamean_y
Temperaturamin_y
Temperaturamax_y
Temperaturastd_y
Ponto de Orvalhomean_y
Ponto de Orvalhomin_y
Ponto de Orvalhomax_y
Ponto de Orvalhostd_y
Velocidade do Ventomean_y
Velocidade do Ventomin_y
Velocidade do Ventomax_y
Velocidade do Ventostd_y
Direção do Ventomean_y
Direção do Ventomin_y
Direção do Ventomax_y
Direção do Ventostd_y
Visibilidademean_y
Visibilidademin_y
Visibilidademax_y
Visibi

In [1]:
import pandas as pd
df = pd.read_csv("data/test_with_flight_v2.csv")

In [2]:
df

,flightid,origem,destino,hora_ref,dt_dep,snapshot_radar,path,hora_esperas,esperas,aero_esperas,...,dt_radar_lambda2,dt_radar_lambda3,flightlevel_mean,flightlevel_max,distancia_,flightlevel_std,flightlevel_std1,speed_std,speed_std1,sum(flight_through_area)
0,4f0356600f61e3fcbea8ed8a137a2423,SBBR,SBSP,11:00:00,11:53:30,MULTIPOINT ((-0.9862962481402141 -0.2329259019...,http://satelite.cptec.inpe.br/repositoriogoes/...,10:00:00,1,SBSP,...,10564.0,844.206687,269.510868,372.424169,834.699654,19.009494,114.817624,87.142962,20.135674,438
1,865dbbbe74bebea18a71f24342516ff0,SBSP,SBCT,00:00:00,00:04:37,MULTIPOINT ((-0.8863534722346053 -0.5093356150...,http://satelite.cptec.inpe.br/repositoriogoes/...,23:00:00,0,SBCT,...,7381.0,334.871433,164.239133,276.821248,334.254521,9.689498,87.120708,81.325769,14.762520,175
2,1803af9cfc6a2c74d188481e3ffd848e,SBGR,SBPA,01:00:00,01:33:15,MULTIPOINT ((-0.8966138193125871 -0.5115119601...,http://satelite.cptec.inpe.br/repositoriogoes/...,00:00:00,0,SBPA,...,9122.0,722.301597,254.844834,351.092908,865.298618,17.465243,116.247042,76.296121,15.923236,617
3,f6af733a687f904183efd149ec713be5,SBCT,SBSV,03:00:00,03:54:14,MULTIPOINT ((-0.8586718860112791 -0.4418648049...,http://satelite.cptec.inpe.br/repositoriogoes/...,02:00:00,0,SBSV,...,8944.0,552.370292,328.999659,383.333333,1752.457895,23.451966,93.777156,67.783456,21.149632,411
4,6117f9ac60b7f66b740c9130be433313,SBRJ,SBGR,00:00:00,00:27:32,MULTIPOINT ((-0.8914635585346382 -0.5234904405...,http://satelite.cptec.inpe.br/repositoriogoes/...,23:00:00,0,SBGR,...,13445.0,423.794971,170.393562,291.021939,344.671385,9.565269,93.363084,76.759758,12.799802,260
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40034,4e08052066073297c7a3c55798618324,SBSV,SBRF,20:00:00,20:25:00,MULTIPOINT ((-0.8955415064673201 -0.5234137952...,http://satelite.cptec.inpe.br/repositoriogoes/...,19:00:00,0,SBRF,...,9182.0,526.762331,226.645097,362.508384,643.117568,15.707042,127.343373,97.043441,14.564714,361
40035,e22f120c65f5759aa8df576b59d71be3,SBCT,SBSP,08:00:00,08:50:22,MULTIPOINT ((-0.8903194203987348 -0.5144905777...,http://satelite.cptec.inpe.br/repositoriogoes/...,07:00:00,0,SBSP,...,6902.0,490.297328,165.953869,276.223053,335.494072,10.369936,86.372285,96.123926,16.439044,104
40036,ca9399162037201a4d0c49c832a32739,SBGR,SBCF,11:00:00,11:30:03,MULTIPOINT ((-0.8963144777278773 -0.5234059876...,http://satelite.cptec.inpe.br/repositoriogoes/...,10:00:00,0,SBCF,...,7620.0,452.974918,201.908039,332.682734,483.495187,12.614499,103.607766,95.569591,15.688727,228
40037,560136088f5f1493dbad809a3554172c,SBBR,SBGR,12:00:00,12:25:21,NaN,http://satelite.cptec.inpe.br/repositoriogoes/...,11:00:00,0,SBGR,...,15064.0,636.873657,263.464528,366.306658,824.563068,16.204102,112.596668,84.884888,17.952150,256


In [88]:
invertedCheckCols(test)

hora_ref
dt_dep
snapshot_radar
path
hora_esperas
aero_esperas
hora_metaf
metaf
aero_metaf
hora_metar
metar
aero_metar
hora_tcp
aero_tcp
hora_tcr
aero_tcr
eqq
data
first(substring(CAST(polygon AS STRING) 3 13))
flight_through_area
aero_x
aero_y
aero
HoraData
linha_first
flightid_
trocareal
troca_prev_meanb2
troca_prev_sumb2
troca_prev_meantcprev_sem
troca_prev_sumtcprev_sem
troca_prev_meanb4
troca_prev_sumb4
troca_prev_meantcreal_hora
troca_prev_sumtcreal_hora
troca_prev_meanb6
troca_prev_sumb6
troca_prev_meantcreal_aero
troca_prev_sumtcreal_aero


In [89]:
checkCols(test)

target
HoraDataDest
troca_prev_mean
troca_prev_sum
troca_realb5
troca_realtcreal_sem
troca_real
